# Perhitungan Sentralitas

## Import Library

In [52]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [53]:
# Pengaturan tampilan
pd.set_option('display.max_colwidth', None)

# Float 8 desimal
pd.options.display.float_format = '{:.8f}'.format

# Membaca CSV
df = pd.read_csv('../data/processed/edges.csv')

# Konversi numerik
df['Weight'] = pd.to_numeric(df['Weight'])
df['Inverse_Weight'] = pd.to_numeric(df['Inverse_Weight'])

# Pembulatan
df['Inverse_Weight'] = df['Inverse_Weight'].round(8)

# Urutkan berdasarkan Weight terbesar
df = df.sort_values(
    by='Weight',
    ascending=False
).reset_index(drop=True)

# Output
display(df)

,Guru,Murid,Weight,Inverse_Weight
0,ابن عمر,نافع,11346,0.00008814
1,معمر,عبد الرزاق,6402,0.00015620
2,ابو هشام بن عروه,هشام بن عروه,6390,0.00015649
3,ابن عباس,عكرمه,5456,0.00018328
4,ابن عباس,سعيد بن جبير,4663,0.00021445
...,...,...,...,...
776793,كيسان,محمد بن ربيعه,1,1.00000000
776794,عليا,صفوان بن عيسي الزهري,1,1.00000000
776795,ناسا من الصحابه,جابر بن زيد,1,1.00000000
776796,الزهري,عبد الرزاق بن همام,1,1.00000000


## Load Data

In [54]:
print(df.dtypes)

Guru                  str
Murid                 str
Weight              int64
Inverse_Weight    float64
dtype: object


In [55]:
df['Weight'] = pd.to_numeric(df['Weight'])
df['Inverse_Weight'] = pd.to_numeric(df['Inverse_Weight'])
print(df.dtypes)

Guru                  str
Murid                 str
Weight              int64
Inverse_Weight    float64
dtype: object


In [56]:
df_subset = df.head(40000)
print(df_subset)

                                 Guru  \
0                             ابن عمر   
1                                معمر   
2                    ابو هشام بن عروه   
3                            ابن عباس   
4                            ابن عباس   
...                               ...   
39995                 ابو سعيد الخدري   
39996         محمد بن اسحاق بن راهويه   
39997                  سواده بن حنظله   
39998  ابو عبد الرحمن بن خالد بن نجيح   
39999                ابو هارون العبدي   

                                         Murid  Weight  Inverse_Weight  
0                                         نافع   11346      0.00008814  
1                                   عبد الرزاق    6402      0.00015620  
2                                 هشام بن عروه    6390      0.00015649  
3                                        عكرمه    5456      0.00018328  
4                                 سعيد بن جبير    4663      0.00021445  
...                                        ...     ...             ..

### Bentuk graph

In [57]:
G = nx.from_pandas_edgelist(
    df_subset,
    source='Murid',
    target='Guru',
    edge_attr=['Weight', 'Inverse_Weight'],
    create_using=nx.DiGraph()
)

In [58]:
print("Jumlah node :", G.number_of_nodes())
print("Jumlah edge :", G.number_of_edges())

Jumlah node : 12169
Jumlah edge : 40000


In [59]:
display(df_subset)

,Guru,Murid,Weight,Inverse_Weight
0,ابن عمر,نافع,11346,0.00008814
1,معمر,عبد الرزاق,6402,0.00015620
2,ابو هشام بن عروه,هشام بن عروه,6390,0.00015649
3,ابن عباس,عكرمه,5456,0.00018328
4,ابن عباس,سعيد بن جبير,4663,0.00021445
...,...,...,...,...
39995,ابو سعيد الخدري,ابو عبد الله بن عبد الرحمن بن ابو صعصعه,7,0.14285714
39996,محمد بن اسحاق بن راهويه,سليمان بن احمد,7,0.14285714
39997,سواده بن حنظله,ابو هلال,7,0.14285714
39998,ابو عبد الرحمن بن خالد بن نجيح,عبد الرحمن بن خالد بن نجيح,7,0.14285714


In [60]:
# Hapus self-loop
G.remove_edges_from(nx.selfloop_edges(G))

In [61]:
print("Jumlah node setelah hapus self-loop:", G.number_of_nodes())
print("Jumlah edge setelah hapus self-loop:", G.number_of_edges())

Jumlah node setelah hapus self-loop: 12169
Jumlah edge setelah hapus self-loop: 40000


In [62]:
# Degree
degree_dict = dict(G.degree(weight='Weight'))
print("✅ Degree selesai")

# Degree Centrality
deg = nx.degree_centrality(G)
print("✅ Degree Centrality selesai.")

# In-Degree Centrality
if G.is_directed():
    in_deg = nx.in_degree_centrality(G)
    print("✅ In-Degree Centrality selesai.")
else:
    in_deg = {}

# Out-Degree Centrality
if G.is_directed():
    out_deg = nx.out_degree_centrality(G)
    print("✅ Out-Degree Centrality selesai.")
else:
    out_deg = {}

✅ Degree selesai
✅ Degree Centrality selesai.
✅ In-Degree Centrality selesai.
✅ Out-Degree Centrality selesai.


In [63]:
# === Eigenvector Centrality (Power Iteration)
try:
    eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=5000, tol=1e-05, weight='Weight')
    print("✅ Eigenvector Centrality selesai.")
except nx.PowerIterationFailedConvergence:
    eigenvector_centrality = {}

# # === Eigenvector Centrality (NumPy version)
# try:
#     eigenvector_numpy_centrality = nx.eigenvector_centrality_numpy(G, weight='Weight')
#     print("✅ Eigenvector Numpy Centrality selesai.")
# except Exception as e:
#     eigenvector_numpy_centrality = {}
#     print(f"Proses gagal: {e}")

✅ Eigenvector Centrality selesai.


In [64]:
from tqdm import tqdm

# fungsi hitung closeness per node
def closeness_for_node(node):
    return node, nx.closeness_centrality(G, u=node, distance="Inverse_Weight")

# daftar semua node
nodes = list(G.nodes())

closeness = {}
for node in tqdm(nodes, desc="Closeness Centrality"):
    closeness[node] = nx.closeness_centrality(G, u=node, distance="Inverse_Weight")


print("✅ Closeness Centrality selesai.")

Closeness Centrality: 100%|██████████| 12169/12169 [1:17:07<00:00,  2.63it/s]

✅ Closeness Centrality selesai.


In [65]:
import networkx as nx
from tqdm import tqdm

def betweenness_centrality_with_progress(G, normalized=True, weight=None):
    nodes = list(G.nodes())
    bet = dict.fromkeys(nodes, 0.0)

    for s in tqdm(nodes, desc="Betweenness Centrality (Nodes)"):
        # hitung betweenness subset: dari source s ke semua node
        contrib = nx.betweenness_centrality_subset(
            G, sources=[s], targets=nodes,
            normalized=normalized, weight=weight
        )
        # gabungkan kontribusi
        for n, v in contrib.items():
            bet[n] += v

    return bet


# =====================
# Pemakaian
# =====================
bet_node = betweenness_centrality_with_progress(G, normalized=True, weight="Inverse_weight")

print("✅ Betweenness selesai.")

Betweenness Centrality (Nodes): 100%|██████████| 12169/12169 [19:15<00:00, 10.53it/s]


✅ Betweenness selesai.


## Menghitung Sentralitas

### Menghitung Degree, Degree Centrality, In Degree, Out Degree, Eigenvector, Closeness, dan Betweenness

In [66]:
# # -----------------------------
# # Menghitung centrality utama
# # -----------------------------

# # Degree
# degree_dict = dict(G.degree())

# # In-Degree (peran sebagai guru)
# in_degree_dict = dict(G.in_degree())

# # Out-Degree (peran sebagai murid)
# out_degree_dict = dict(G.out_degree())

# # Degree Centrality
# degree_centrality_dict = nx.degree_centrality(G)

# # Eigenvector Centrality
# try:
#     eigenvector_centrality = nx.eigenvector_centrality(
#         G,
#         max_iter=5000,
#         tol=1e-06,
#         weight='Weight'
#     )
#     print("Eigenvector Centrality selesai.")
# except nx.PowerIterationFailedConvergence as e:
#     eigenvector_centrality = {}
#     print(f"Eigenvector tidak konvergen: {e}")
# except Exception as e:
#     eigenvector_centrality = {}
#     print(f"Eigenvector gagal: {e}")

# eigenvector_dict = eigenvector_centrality

# # Closeness Centrality
# closeness_dict = nx.closeness_centrality(
#     G,
#     distance='Inverse_Weight'
# )

# # Betweenness Centrality
# betweenness_dict = nx.betweenness_centrality(
#     G,
#     normalized=True,
#     weight='Inverse_Weight'
# )

# # Edge Betweenness Centrality
# edge_betweenness = nx.edge_betweenness_centrality(
#     G,
#     normalized=True,
#     weight='Inverse_Weight'
# )

# # Simpan atribut ke node
# for node in G.nodes():
#     G.nodes[node]['degree'] = degree_dict.get(node, 0)
#     G.nodes[node]['degree_centrality'] = degree_centrality_dict.get(node, 0)
#     G.nodes[node]['in_degree'] = in_degree_dict.get(node, 0)
#     G.nodes[node]['out_degree'] = out_degree_dict.get(node, 0)
#     G.nodes[node]['eigenvector_centrality'] = eigenvector_centrality.get(node, 0)
#     G.nodes[node]['closeness_centrality'] = closeness_dict.get(node, 0)
#     G.nodes[node]['betweenness_centrality'] = betweenness_dict.get(node, 0)

# # Simpan atribut ke edge
# for u, v in G.edges():
#     G.edges[u, v]['betweenness_centrality'] = edge_betweenness.get((u, v), 0)

# # Gabungkan semua hasil ke DataFrame
# centrality_df = pd.DataFrame({
#     'Perawi': list(G.nodes()),
#     'Degree': [degree_dict.get(node, 0) for node in G.nodes()],
#     'Degree_Centrality': [degree_centrality_dict.get(node, 0) for node in G.nodes()],
#     'In_Degree': [in_degree_dict.get(node, 0) for node in G.nodes()],
#     'Out_Degree': [out_degree_dict.get(node, 0) for node in G.nodes()],
#     'Eigenvector_Centrality': [eigenvector_centrality.get(node, 0) for node in G.nodes()],
#     'Closeness_Centrality': [closeness_dict.get(node, 0) for node in G.nodes()],
#     'Betweenness_Centrality': [betweenness_dict.get(node, 0) for node in G.nodes()]
# })

# centrality_df = centrality_df.sort_values(
#     by='Betweenness_Centrality',
#     ascending=False
# ).reset_index(drop=True)

# # Tampilkan 20 teratas
# print("=== Top 20 per Betweenness Centrality ===")
# display(centrality_df.head(20))

# # # Simpan ke CSV
# # output_path = '../data/processed/centrality_all.csv'
# # centrality_df.to_csv(output_path, index=False, encoding='utf-8-sig')
# # print(f'Hasil centrality disimpan ke: {output_path}')

In [67]:
# nx.write_graphml(G_fixed, '../data/processed/CentralityPerawi.graphml')